In [10]:
import os, sys, json
import pandas as pd
from google.cloud import bigquery
PROJECT_ID = "project-5eb4889f-74fb-4ad5-a76"
LOCATION   = "EU"

creds = None

if "google.colab" in sys.modules:
    from google.colab import auth
    auth.authenticate_user()
elif os.environ.get("GCP_SA_KEY"):
    from google.oauth2 import service_account
    creds = service_account.Credentials.from_service_account_info(
        json.loads(os.environ["GCP_SA_KEY"]),
        scopes=["https://www.googleapis.com/auth/cloud-platform"])

client = bigquery.Client(project=PROJECT_ID, location=LOCATION, credentials=creds)
print("проєкт:", client.project)

проєкт: project-5eb4889f-74fb-4ad5-a76


In [21]:
import papermill as pm
import uuid

NOTEBOOKS = [
    "01_bronze.ipynb",
    "02_dim_currency.ipynb",
    "03_dim_date.ipynb",
    "04_snapshot.ipynb",
    "05_fact_merge.ipynb",
]

os.makedirs("runs", exist_ok=True)
print("робоча тека:", os.getcwd())

робоча тека: /Users/ik/nbu-pipeline


In [22]:
run_id = str(uuid.uuid4())      # один на весь прогін
log_rows = []

for step_no, name in enumerate(NOTEBOOKS, start=1):
    started = pd.Timestamp.now(tz="UTC")
    try:
        pm.execute_notebook(name, f"runs/{name}", kernel_name="python3")
        status, error = "OK", ""
    except Exception as e:
        status, error = "FAILED", f"{type(e).__name__}: {e}"[:1000]
    finished = pd.Timestamp.now(tz="UTC")

    log_rows.append({
        "run_id": run_id,
        "step_no": step_no,
        "notebook": name,
        "status": status,
        "started_at": started,
        "finished_at": finished,
        "duration_sec": (finished - started).total_seconds(),
        "error": error,
    })
    print(f"{step_no}. {name}: {status}")

    # кожен наступний ноутбук читає результат попереднього, тому зупиняємось
    if status == "FAILED":
        print(f"Зупинились на ноутбуці {name}, решту не запускаємо")
        break

log_df = pd.DataFrame(log_rows)
log_df

Executing:   0%|          | 0/7 [00:00<?, ?cell/s]

[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


1. 01_bronze.ipynb: OK


Executing:   0%|          | 0/9 [00:00<?, ?cell/s]

[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


...

4. 04_snapshot.ipynb: OK


Executing:   0%|          | 0/7 [00:00<?, ?cell/s]

[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


5. 05_fact_merge.ipynb: OK


,run_id,step_no,notebook,status,started_at,finished_at,duration_sec,error
0,fb47585f-675f-4170-b62c-38111ff15fee,1,01_bronze.ipynb,OK,2026-09-28 18:56:00.899253+00:00,2026-09-28 18:56:06.659190+00:00,5.759937,
1,fb47585f-675f-4170-b62c-38111ff15fee,2,02_dim_currency.ipynb,OK,2026-09-28 18:56:06.659841+00:00,2026-09-28 18:56:12.539878+00:00,5.880037,
2,fb47585f-675f-4170-b62c-38111ff15fee,3,03_dim_date.ipynb,OK,2026-09-28 18:56:12.540131+00:00,2026-09-28 18:56:22.276195+00:00,9.736064,
3,fb47585f-675f-4170-b62c-38111ff15fee,4,04_snapshot.ipynb,OK,2026-09-28 18:56:22.276376+00:00,2026-09-28 18:56:34.011702+00:00,11.735326,
4,fb47585f-675f-4170-b62c-38111ff15fee,5,05_fact_merge.ipynb,OK,2026-09-28 18:56:34.011830+00:00,2026-09-28 18:56:43.121523+00:00,9.109693,


In [25]:
ds_meta = bigquery.Dataset(f"{PROJECT_ID}.nbu_meta")
ds_meta.location = "EU"
client.create_dataset(ds_meta, exists_ok=True)

log_schema = [
    bigquery.SchemaField("run_id",       "STRING"),
    bigquery.SchemaField("step_no",      "INTEGER"),
    bigquery.SchemaField("notebook",     "STRING"),
    bigquery.SchemaField("status",       "STRING"),
    bigquery.SchemaField("started_at",   "TIMESTAMP"),
    bigquery.SchemaField("finished_at",  "TIMESTAMP"),
    bigquery.SchemaField("duration_sec", "FLOAT"),
    bigquery.SchemaField("error",        "STRING"),
]

LOG_ID = f"{PROJECT_ID}.nbu_meta.run_log"

table = bigquery.Table(LOG_ID, schema=log_schema)
table.time_partitioning = bigquery.TimePartitioning(field="started_at")
client.create_table(table, exists_ok=True)

# WRITE_APPEND: журнал накопичується, старі запуски не стираємо
cfg = bigquery.LoadJobConfig(schema=log_schema, write_disposition="WRITE_APPEND")
client.load_table_from_dataframe(log_df, LOG_ID, job_config=cfg).result()
print("записано в журнал:", len(log_df))

/Users/ik/nbu-pipeline/.venv/lib/python3.13/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


записано в журнал: 5


In [26]:
ok = (log_df["status"] == "OK").sum()
failed = (log_df["status"] == "FAILED").sum()
print(f"успішно: {ok}, впало: {failed}, "
      f"загальна тривалість: {log_df['duration_sec'].sum():.1f} с")

print(client.query(f"""
    SELECT run_id, step_no, notebook, status, duration_sec
    FROM `{PROJECT_ID}.nbu_meta.run_log`
    ORDER BY started_at DESC
    LIMIT 10
""").to_dataframe())

успішно: 5, впало: 0, загальна тривалість: 42.2 с


/Users/ik/nbu-pipeline/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


                                 run_id  step_no               notebook  \
0  fb47585f-675f-4170-b62c-38111ff15fee        5    05_fact_merge.ipynb   
1  fb47585f-675f-4170-b62c-38111ff15fee        4      04_snapshot.ipynb   
2  fb47585f-675f-4170-b62c-38111ff15fee        3      03_dim_date.ipynb   
3  fb47585f-675f-4170-b62c-38111ff15fee        2  02_dim_currency.ipynb   
4  fb47585f-675f-4170-b62c-38111ff15fee        1        01_bronze.ipynb   

  status  duration_sec  
0     OK      9.109693  
1     OK     11.735326  
2     OK      9.736064  
3     OK      5.880037  
4     OK      5.759937  
